## Load Libraries

In [22]:
import os                     # Work with environment variables and file paths
import requests               # Send HTTP requests 
import glob                   # Find files using wildcard patterns
import subprocess             # Run external commands or shell processes
from dotenv import load_dotenv    # Load environment variables

from openai import OpenAI                                       # For interacting with the OpenAI API

from langchain_openai import ChatOpenAI                         # For interacting with the OpenAI API via langchain
from langchain_ollama import ChatOllama                         # For interacting with the Ollama API via langchain

from langchain_core.messages import SystemMessage, HumanMessage # Create prompts

from langchain_openai import OpenAIEmbeddings                   # Create embeddings using OpenAI models
from langchain_chroma import Chroma                             # Chroma vector database integration for LangChain
from langchain_huggingface import HuggingFaceEmbeddings         # Create embeddings using HuggingFace models

from langchain_community.document_loaders import DirectoryLoader # load many files from a folder
from langchain_community.document_loaders import TextLoader # load text files into LangChain documents
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Splits large documents into smaller chunks for embedding

import gradio as gr # User interface

from IPython.display import Markdown, display  # Display formatted Markdown output in Jupyter notebooks

import re  # Regular expressions for pattern matching in text

## Load Environment Key

In [2]:
try:
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    script_dir = os.getcwd()

# Go up one level to the project folder
project_dir = os.path.dirname(script_dir)
env_path = os.path.join(project_dir, "env_keys", ".env")

# Access the variable
load_dotenv(dotenv_path=env_path)
openai_api_key = os.getenv("OPENAI_API_KEY")
print("API Key loaded:", openai_api_key is not None)

API Key loaded: True


## Ollma Initialize

In [3]:
subprocess.Popen("ollama serve", shell=True)

<Popen: returncode: None args: 'ollama serve'>

In [4]:
requests.get("http://localhost:11434").content

b'Ollama is running'

In [5]:
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(result.stdout)

NAME                ID              SIZE      MODIFIED    
qwen2.5-coder:7b    dae161e27b0e    4.7 GB    4 weeks ago    
llama3.2:latest     a80c4f17acd5    2.0 GB    4 weeks ago    



## Configuration

In [14]:
model_openai = "gpt-4.1-mini"
model_ollma = "llama3.2"

db_name = "vector_db"

## Character Text Split

In [18]:
files = glob.glob("knowledge-base/*")

documents = []

for file_path in files:
    doc_type = os.path.basename(file_path)
    loader = DirectoryLoader(path=file_path, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for docs in folder_docs:
        docs.metadata['doc_type'] = doc_type
        documents.append(docs)
        
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

## Make Vector Database and Store

In [15]:
# Encorder model for vector embeddings
embedding = HuggingFaceEmbeddings(model='all-MiniLM-L6-v2')

In [20]:
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embedding).delete_collection()

vector_store = Chroma.from_documents(documents=chunks, embedding=embedding, persist_directory=db_name)
print(f"Vector store created with {vector_store._collection.count()} documents")

Vector store created with 413 documents


## LangChain LLM and Retriver

In [24]:
retriver = vector_store.as_retriever()

In [56]:
llm = ChatOllama(temperature=0, model=model_ollma)

In [58]:
retriver.invoke("Who is James")

[Document(id='d9847712-0f92-4edd-b572-032cc58b172d', metadata={'source': 'knowledge-base\\contracts\\Contract with Guardian Life Partners for Lifellm.md', 'doc_type': 'contracts'}, page_content="_________________________________\n**Jonathan Park**\n**Title**: President & CEO\n**Guardian Life Partners**\n**Date**: March 1, 2025\n\n---\n\n*This contract establishes Guardian Life Partners as a strategic partner leveraging Lifellm's advanced AI underwriting and digital health integration to modernize life insurance operations.*"),
 Document(id='259d3f84-76ee-46ef-8d2c-f7c1f1db2590', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Robert Chen.md'}, page_content="## Other HR Notes\n- **Education:** MS in Computer Science from Stanford University, BS in Computer Engineering from MIT\n- **Technical Expertise:** Expert in React, Node.js, TypeScript, PostgreSQL, AWS, microservices architecture\n- **Recognition:** Engineering Excellence Award 2023, Technical Leadership Aw

## Prompt

In [59]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

## Calling RAG and LLM

In [66]:
def answer_question(questions: str, history):
    docs = retriver.invoke(questions)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke(input=[SystemMessage(content=system_prompt), HumanMessage(content=questions)])
    return response.content

In [64]:
Markdown(answer_question(questions="Who is Bishop and his title?"))

Jordan K. Bishop is a valued member of the Insurellm family, and his current title is Frontend Software Engineer.

## Gradio UI

In [73]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
